[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week12_capstone_STARTER.ipynb)


# Week 12 -- The Capstone (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** HAM10000 skin lesion capstone — full pipeline: clean → classify → XAI → audit → model card

**Dataset:** HAM10000 skin lesion (capstone)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


In [ ]:
# -- Monday: Write the Problem Statement BEFORE Loading Any Data -----------
# This cell MUST be completed before running any other cell this week.
# Writing the problem statement first prevents confirmation bias:
# you cannot later choose the metric that makes your model look best.

PROBLEM_STATEMENT = {
    # What are we predicting?
    'task': (
        'Multi-class skin lesion classification: predict 1 of 7 diagnostic '
        'categories (nv, mel, bkl, bcc, akiec, df, vasc) from a dermoscopy image.'
    ),

    # Primary metric -- commit BEFORE training
    # WHY macro_auc: threshold-free, treats all 7 classes equally,
    # correctly handles rare classes where F1@0.5 collapses to near-zero
    'primary_metric': 'macro_auc',
    'primary_threshold': 0.85,   # deployment requirement

    # Secondary metric -- the most clinically critical single number
    # Missed melanoma (false negative) is life-threatening
    'secondary_metric': 'melanoma_recall',
    'secondary_threshold': 0.80,  # 80% sensitivity minimum

    # Known risks -- list at least 3 before looking at the data
    'risks': [
        'Class imbalance: Melanocytic nevus is ~67% of the dataset.',
        'Age bias: elderly patients have atypical presentations.',
        'Duplicate images: same lesion appears multiple times -- split by lesion_id.',
        'Instrument variation: dx_type column encodes diagnostic confidence.',
    ],

    'committed_date': '2026-06-08',   # update to today
    'committed_by':   'Ngozi Eze-Okafor',
}

print('=' * 60)
print('PROBLEM STATEMENT -- HAM10000 Capstone')
print('=' * 60)
print(f"Task:            {PROBLEM_STATEMENT['task'][:60]}...")
print(f"Primary metric:  {PROBLEM_STATEMENT['primary_metric']} > {PROBLEM_STATEMENT['primary_threshold']}")
print(f"Secondary metric:{PROBLEM_STATEMENT['secondary_metric']} > {PROBLEM_STATEMENT['secondary_threshold']}")
print(f"Risks identified:{len(PROBLEM_STATEMENT['risks'])}")
print('=' * 60)
print('You may now load the data. You may NOT change the thresholds above.')


In [ ]:
# -- Tuesday: Remove Duplicates by lesion_id -- Prevent Data Leakage ------
# HAM10000 contains multiple images of the SAME lesion (different angles,
# zoom levels, lighting). If the same lesion appears in both train and test,
# the model memorises lesion-specific appearance -- not general features.
# This inflates test metrics by 5-15%. Fix: split by lesion_id.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load metadata CSV (columns: image_id, lesion_id, dx, age, sex, dx_type)
try:
    meta = pd.read_csv('datasets/ham10000/HAM10000_metadata.csv')
    print(f'Loaded: {len(meta)} images')
except FileNotFoundError:
    # Synthetic metadata for structure demonstration
    n = 10015
    meta = pd.DataFrame({
        'image_id':  [f'ISIC_{i:07d}' for i in range(n)],
        'lesion_id': [f'HAM_{i//3:05d}' for i in range(n)],  # ~3 imgs/lesion
        'dx': np.random.choice(['nv','mel','bkl','bcc','akiec','df','vasc'],
                               n, p=[0.670,0.112,0.110,0.051,0.033,0.012,0.012]),
        'age': np.random.choice([20,30,40,50,60,70,80], n),
        'sex': np.random.choice(['male','female'], n),
        'dx_type': np.random.choice(['histo','follow_up','consensus'], n),
    })
    print(f'Synthetic metadata: {n} images')

# Class distribution
print('\nClass distribution:')
for dx, count in meta['dx'].value_counts().items():
    pct = 100 * count / len(meta)
    bar = '#' * int(pct / 2)   # ASCII bar chart
    print(f'  {dx:6} {bar:<35} {count:5} ({pct:.1f}%)')

# Split by lesion_id (NOT by image)
unique_lesions = meta['lesion_id'].unique()

# One diagnosis per lesion (all images from one lesion have the same dx)
lesion_dx = meta.drop_duplicates('lesion_id').set_index('lesion_id')['dx']

# 80% train, 20% test at the LESION level, stratified by class
train_lesions, test_lesions = train_test_split(
    unique_lesions, test_size=0.20, random_state=SEED,
    stratify=lesion_dx[unique_lesions])   # maintain class balance

# Select images belonging to each split
train_meta = meta[meta['lesion_id'].isin(train_lesions)]
test_meta  = meta[meta['lesion_id'].isin(test_lesions)]

print(f'\nTrain: {len(train_lesions)} lesions | {len(train_meta)} images')
print(f'Test:  {len(test_lesions)} lesions  | {len(test_meta)} images')

# Verify zero leakage: no lesion appears in both splits
leakage = set(train_lesions) & set(test_lesions)
assert len(leakage) == 0, f'DATA LEAKAGE: {len(leakage)} shared lesions!'
print('Leakage check PASSED -- zero shared lesions')

# Save for subsequent cells
train_meta.to_csv('outputs/ham10000_train_meta.csv', index=False)
test_meta.to_csv('outputs/ham10000_test_meta.csv', index=False)
print('Saved split metadata to outputs/')


## Learning Objectives

By the end of this week, you will be able to:

- Apply the complete twelve-week pipeline to a new dataset (HAM10000) without step-by-step guidance
- Make independent decisions about preprocessing, model choice, evaluation, and XAI method
- Conduct a required fairness audit — even though the brief does not explicitly ask for one
- Deliver an executive brief in Pyramid Principle order and a clinical XAI report
- Complete your internship portfolio and write the portfolio letter



---

## MONDAY -- "Problem Statement and First Look"


Write the problem statement before you load any data. What are we predicting? Why does it matter? Which error type is most costly — missed melanoma (false negative) or unnecessary biopsy (false positive)? What could go wrong? Write at least three risks. Then load the data.


### Exercise 12.1 -- Problem statement

Write the problem statement markdown cell BEFORE loading any data. Include: what, why, primary metric, deployment threshold, and at least THREE risks. Sign it with today's date. You cannot change it after loading the data.


In [ ]:
# Exercise 12.1: Problem statement
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Problem Statement and First Look (scaffold) --
# Problem statement markdown cell — WRITE BEFORE RUNNING ANY CODE
# 1. What: multi-class skin lesion classification, 7 categories
# 2. Why: dermoscopy AI can assist triage in settings with limited dermatologist access
# 3. Primary metric: macro AUC (class-balanced, clinically motivated)
# 4. Deployment threshold: macro AUC > 0.85, Melanoma recall > 0.80
# 5. Risks:
#    (a) Class imbalance — Melanocytic nevus is 10x more common than Vascular lesion
#    (b) Age bias — elderly patients have more atypical presentations
#    (c) Instrument variation — dermoscope model affects image quality
#    (d) Missing metadata — not all patients have age recorded


### Monday Morning Moment

*Slack — Monday, 9:00am.*

**Yewande Adeyemi:** Ngozi. Did you write the problem statement before loading the data?

**Ngozi Eze-Okafor:** Yes. Primary metric: macro AUC. Deployment threshold: macro AUC > 0.85 and Melanoma recall > 0.80. Three risks: class imbalance, age bias, and instrument variation.

**Yewande Adeyemi:** You sound like Dr. Kwame.

**Ngozi Eze-Okafor:** I've been here twelve weeks.

**Dr. Kwame Asante:** The instrument variation risk — how will you detect it?

**Ngozi Eze-Okafor:** HAM10000 has a dx_type column — histopathology confirmed vs follow-up consensus vs confocal microscopy. I can use that as a proxy for diagnostic confidence. Images with lower-confidence labels may have worse model performance.

**Dr. Kwame Asante:** Add it to your fairness audit dimensions. You found it.




---

## TUESDAY -- "Pipeline: Clean → Augment → Train"


Apply your preprocessing module. HAM10000 has duplicate images (same lesion, different zoom). Remove duplicates before splitting. HAM10000 is highly imbalanced — Melanocytic nevus dominates. Choose your strategy before training: oversample, undersample, or weight the loss. Document the decision.


### Exercise 12.2 -- Duplicate removal

HAM10000 contains duplicate images (multiple views of the same lesion). Use the lesion_id column in the metadata to identify and remove duplicates before splitting. How many unique lesions are there vs total images? What is the effect of keeping duplicates on the train/test split?


In [ ]:
# Exercise 12.2: Duplicate removal
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Pipeline: Clean → Augment → Train (scaffold) --
# Key decisions to document:
# 1. Duplicate removal strategy
# 2. Class imbalance strategy (oversample/undersample/weight)
# 3. Augmentation choices — which are valid for dermoscopy?
#    (horizontal flip: YES — lesions have no orientation)
#    (colour jitter: careful — colour is diagnostic in dermoscopy)
# 4. Model choice — EfficientNet-B0 (Week 8), ResNet-18 (Week 7), or new?
# 5. Why that model for this dataset?



---

## WEDNESDAY -- "XAI — GradCAM on Melanoma Misclassifications"


Run GradCAM on at least 5 misclassified Melanoma images (predicted as benign). For each: where did the model look? Is the highlighted region the lesion itself, or a confound (hair, ruler, ink mark)? This is the clinical deliverable — what can a dermatologist learn from these heatmaps?


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: XAI — GradCAM on Melanoma Misclassifications (scaffold) --
# Melanoma misclassification XAI
melanoma_false_negatives = [
    (img, true_label, pred_label, pred_prob)
    for img, true_label, pred_label, pred_prob in zip(test_images, y_test, y_pred, y_proba)
    if true_label == MELANOMA_IDX and pred_label != MELANOMA_IDX
]
print(f"Melanoma false negatives: {len(melanoma_false_negatives)}")

# For each: generate GradCAM + write clinical annotation
# "The model predicted [class] with [prob]% confidence.
#  GradCAM shows the model attended to [region].
#  The lesion itself is in [other region].
#  Clinical consequence: this patient's melanoma would not be flagged for biopsy."



---

## THURSDAY -- "Fairness Audit and Executive Brief"


HAM10000 includes patient_age. Compute per-age-group performance: AUC for patients < 40, 40-60, 60-80, and > 80. Is the model less accurate for elderly patients? (Elderly skin lesions may appear atypical; the training set may have fewer elderly patients.) Write the executive brief in Pyramid Principle order.


### Exercise 12.3 -- Age fairness audit

Compute per-age-group macro AUC. Is the gap between the youngest and oldest group statistically significant? Run a bootstrap confidence interval for the AUC gap. Write the fairness finding as one sentence Nurse Folake can read.


In [ ]:
# Exercise 12.3: Age fairness audit
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Fairness Audit and Executive Brief (scaffold) --
# Age-group fairness audit
age_bins = {"<40": (0,40), "40-60": (40,60), "60-80": (60,80), ">80": (80,120)}
for label, (lo, hi) in age_bins.items():
    mask = (meta["age"] >= lo) & (meta["age"] < hi)
    if mask.sum() > 50:
        auc = roc_auc_score(y_true[mask], y_proba[mask], multi_class="ovr", average="macro")
        n   = mask.sum()
        print(f"  {label:<8}: macro AUC = {auc:.4f} (n={n})")



---

## FRIDAY -- "The Reflection and the Portfolio Letter"


Four questions. Write them carefully. (1) What did you find hardest this week and how did you handle it? (2) What would you do differently if you started Week 0 again? (3) What is the single most important thing you learned in Book 3 that you did not know before Week 0? (4) What is the first thing you will do when the next medical imaging project arrives? Then write the portfolio letter.


### Exercise 12.4 -- Portfolio letter

Write the portfolio letter in the final cell of capstone.ipynb. Address it "To whoever reads this." Include: what the notebook contains, the fairness finding you are most proud of, one thing you would change, one clinical consequence you caught, and one unanswered question you leave behind.


In [ ]:
# Exercise 12.4: Portfolio letter
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Reflection and the Portfolio Letter (scaffold) --
# Portfolio letter — final cell of capstone.ipynb
# Address: "To whoever reads this"
# Include:
#   - What this notebook contains
#   - The fairness finding you are most proud of documenting honestly
#   - One thing you would change
#   - One clinical consequence you are glad you caught
#   - The question you leave unanswered — and why it matters


### Friday Workplace Moment

*MediVision AI — Friday, 4:30pm. Ngozi submits.*

**Dr. Kwame Asante:** Portfolio letter. Read the most important sentence.

**Ngozi Eze-Okafor:** "The fairness audit showed that patients over 80 have 11 percentage points lower Melanoma AUC than patients under 40. I did not hide this finding. It is the first number in the executive brief."

**Dr. Kwame Asante:** That sentence — not the AUC, not the architecture, not the GradCAM. That sentence is what twelve weeks produced.

**Nurse Folake Balogun:** Can we use this model?

**Ngozi Eze-Okafor:** With monitoring. Macro AUC is 0.87 — above threshold. Melanoma recall is 0.83 — above threshold. But the elderly patient gap requires a retraining commitment in 6 months with additional cases from patients over 80.

**Dr. Kwame Asante:** Conditional approval. Write the deployment recommendation. Twelve weeks, Ngozi.

**Ngozi Eze-Okafor:** Twelve weeks.



## YOUR WEEK 12 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I wrote the problem statement before loading any data, including at least three risks.
- [ ] I removed duplicates by lesion_id before splitting — not after.
- [ ] I conducted the age-group fairness audit even though the brief did not ask for it.
- [ ] My executive brief leads with findings; the methodology is in the appendix.
- [ ] The portfolio letter is addressed to "whoever reads this" — not to Dr. Kwame.


## Challenge Task

> **Core path:** Assemble the complete internship portfolio: all notebooks clean and executable, model cards for all five models (Pets CNN, Chest X-ray, BreakHis, TCGA Uterine, HAM10000), fairness briefs for all three fairness audits, and a 500-word retrospective comparing your Week 1 domain characterisation to how you would approach it now.

> **Advanced path:** Answer the fairness impossibility theorem empirically: using the HAM10000 dataset, show that you cannot simultaneously satisfy calibration, false positive rate parity, and false negative rate parity across age groups when the melanoma base rates differ. Which constraint is violated in your model?


## Common Mistakes

**Splitting HAM10000 by image instead of by lesion_id:** Multiple images of the same lesion appear in both train and test. The model memorises lesion-specific appearance and reports inflated AUC.

**Skipping the fairness audit because the brief did not ask for it:** Week 11 taught you that the fairness audit is always required. A brief that does not mention it does not excuse the omission.

**Treating the portfolio letter as a formality:** The portfolio letter is the document that tells a hiring manager what you know and what you learned. Write it carefully. It is the most professionally important document in the notebook.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
